# PC 게임패드 + 유선 LAN 수집
게임패드는 브라우저를 여는 **PC**에, USB·CSI 카메라와 로봇팔은 **Pi**에 연결합니다.
이 노트북은 Pi 호스트의 `.venv` 커널에서 실행합니다. 제조사 Docker 노트북이 아닙니다.
먼저 [PC_LAN.md](PC_LAN.md)의 Jupyter/SSH 터널 설정을 마치세요.
서버 실행만으로 모터는 움직이지 않습니다. 별도 수집 화면에서 시작 버튼을 누릅니다.


In [ ]:
from pathlib import Path
import sys, importlib.util
repo = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "school_pdf/data_collect.py").exists()), None)
assert repo is not None, "이 저장소 안에서 노트북을 여세요"
for name in ["Arm_Lib", "cv2", "picamera2"]:
    assert importlib.util.find_spec(name), f"{name} 없음: Pi .venv 커널을 선택하세요"
print("Pi Python:", sys.executable)
print("Repository:", repo)


## 수집 서버 실행
기존 Pi 수집 서비스와 제조사 카메라 앱은 먼저 종료하세요. 이미 열린 서버를 덮어쓰지 않습니다.

In [ ]:
import subprocess, urllib.request, urllib.error, time
try:
    urllib.request.urlopen("http://127.0.0.1:8081/status", timeout=2)
except urllib.error.URLError:
    if "server" in globals() and server.poll() is None:
        raise RuntimeError("서버가 이미 시작 중입니다. 아래 상태 셀을 실행하세요")
    log_path = repo / "school_pdf/pc_collect.log"
    with log_path.open("a") as log:
        server = subprocess.Popen(
            [sys.executable, "-u", str(repo / "school_pdf/data_collect.py"), "--input", "pc"],
            cwd=repo, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    print("서버 시작 PID:", server.pid, "로그:", log_path)
else:
    raise RuntimeError("8081 포트에 서버가 있습니다. 기존 서버를 먼저 종료하세요")


## 준비 상태와 조작 화면
시작 직후라면 몇 초 후 이 셀을 다시 실행하세요. 아래 링크는 **PC 브라우저**에서 엽니다. 조작 중에는 해당 탭을 활성화합니다.

In [ ]:
import json
from IPython.display import display, HTML
try:
    state = json.load(urllib.request.urlopen("http://127.0.0.1:8081/status", timeout=3))
    print(json.dumps(state, ensure_ascii=False, indent=2))
except urllib.error.URLError:
    print("서버 초기화 중 또는 실패. 로그:")
    print(log_path.read_text()[-4000:])
display(HTML('<a href="http://localhost:18081" target="_blank">PC 게임패드 수집 화면 열기</a>'))


## 사용
1. 수집 화면에서 게임패드 버튼을 한번 누르고 축·버튼 값을 확인합니다.
2. 두 카메라가 보이면 **수집 시작**을 누릅니다. 스틱·버튼을 모두 놓은 뒤 조작합니다.
3. **START(9)** 또는 웹 정지로 저장을 종료합니다. **SELECT(8)는 초기 자세 이동**입니다.
4. LAN 단절/탭 비활성/게임패드 분리 시 정지합니다. 재연결 후 시작 버튼을 다시 눌러야 합니다.
데이터는 Pi `school_pdf/datasets/`에 저장합니다. CSV 각도는 목표각이고 두 카메라는 동시 노출이 아닙니다.


## 서버 종료
이 노트북에서 띄운 서버만 종료합니다. 다시 조작하려면 서버 실행 셀부터 시작하세요.

In [ ]:
if "server" in globals() and server.poll() is None:
    server.terminate()
    try:
        server.wait(timeout=10)
        print("서버 종료 완료")
    except subprocess.TimeoutExpired:
        print("종료 지연: 장비 옆에서 상태를 확인하고 로그를 확인하세요")
else:
    print("이 노트북이 실행한 서버가 없습니다")
